# **Assignment 13**

In [3]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import HTML
import joblib
import re
import torch
import torch.nn as nn
import random
import torch.nn.functional as F
from sklearn.metrics import f1_score
import torch.optim as optim
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from itertools import product
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

random_seed = 42


In [4]:
def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)

        # Get rid of trailing whitespace
        df.columns = df.columns.str.strip()           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined


def split_csvfiles(datafolder, random_seed, training_prop, validation_prop):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)

    random.seed(random_seed)
    random.shuffle(csv_files)

    train_n = int(len(csv_files) * training_prop)
    val_n = int(len(csv_files) * validation_prop)

    # Split
    if validation_prop == 0:
        train_files = csv_files[:train_n]
        test_files = csv_files[train_n:]

        return train_files, test_files

    else:
        train_files = csv_files[:train_n]
        val_files = csv_files[train_n: train_n + val_n]
        test_files = csv_files[train_n + val_n:]

        return train_files, val_files, test_files
    
def get_middle_frames(x_3d, n_frames_each_side=10):
    """
    For each sample, find the non-zero frames, then take n_frames_each_side
    frames around the center of that non-zero region.
    
    Args:
        x_3d: numpy array of shape (n_samples, n_frames, n_features)
        n_frames_each_side: number of frames to take on each side of center
    
    Returns:
        numpy array of shape (n_samples, 2 * n_frames_each_side, n_features)
    """
    n_samples = x_3d.shape[0]
    n_out_frames = 2 * n_frames_each_side
    result = np.zeros((n_samples, n_out_frames, x_3d.shape[2]))

    for i in range(n_samples):
        # A frame is "active" if any feature in it is non-zero
        nonzero_frames = np.where(x_3d[i].any(axis=1))[0]
        
        if len(nonzero_frames) == 0:
            print(f"Warning: sample {i} has no non-zero frames")
            continue
        
        # Center of the actual motion
        first_frame = nonzero_frames[0]
        last_frame  = nonzero_frames[-1]
        center = (first_frame + last_frame) // 2
        
        start = center - n_frames_each_side
        end   = center + n_frames_each_side

        # Clamp to valid range
        start_clamped = max(0, start)
        end_clamped   = min(x_3d.shape[1], end)

        # Place into output (handles edge case where clamping shifts the window)
        out_start = start_clamped - start
        out_end   = out_start + (end_clamped - start_clamped)

        result[i, out_start:out_end, :] = x_3d[i, start_clamped:end_clamped, :]

    print(f"Output shape: {result.shape}")
    return result




def reshape_to_2d(x, feature_columns):
    """
    Reshape from (n_samples, n_frames * n_features) to (n_samples, n_frames, n_features).
    
    Args:
        x: numpy array of shape (n_samples, n_frames * n_features)
        feature_columns: list of column names (excluding 'target')
    
    Returns:
        numpy array of shape (n_samples, n_frames, n_features)
    """
    # Extract unique frame numbers and sort them
    frame_numbers = sorted(set(
        int(re.search(r'frame(\d+)_', col).group(1))
        for col in feature_columns
    ))
    
    # Extract unique feature names (e.g. 'head_x', 'left_shoulder_y', ...)
    feature_names = sorted(set(
        re.sub(r'^frame\d+_', '', col)
        for col in feature_columns
    ))
    
    n_samples = x.shape[0]
    n_frames = len(frame_numbers)
    n_features = len(feature_names)
    
    print(f"Reshaping: {n_samples} samples, {n_frames} frames, {n_features} features per frame")
    
    # Build index mapping: for each (frame, feature) find the column index
    col_index = {col: i for i, col in enumerate(feature_columns)}
    
    x_3d = np.zeros((n_samples, n_frames, n_features))
    
    for f_idx, frame_num in enumerate(frame_numbers):
        for feat_idx, feat_name in enumerate(feature_names):
            col_name = f"frame{frame_num}_{feat_name}"
            if col_name in col_index:
                x_3d[:, f_idx, feat_idx] = x[:, col_index[col_name]]
    
    return x_3d



## Load cut and padded squat sequences with target determining if squat good or bad

In [5]:
# padded videos with target 0 or 1 based on if it's a bad or good squat
data_folder = "../../MainProject/data/mediapipe_padded_videos"


# Split which files should be train, val and test
training_proportion = 0.9
validation_proportion = 0

train_files, test_files = split_csvfiles(data_folder, random_seed, training_proportion, validation_proportion)

# Load df from files selected to be either training or testing
train_data = load(train_files, data_folder)
test_data = load(test_files, data_folder)

# Split
y_train = train_data["target"].values
x_train = train_data.drop(columns="target").values

y_test = test_data["target"].values
x_test = test_data.drop(columns="target").values

feature_columns = list(train_data.drop(columns="target").columns)

# Reshape to (n_samples, n_frames, n_features)
x_train_3d = reshape_to_2d(x_train, feature_columns)
x_test_3d  = reshape_to_2d(x_test,  feature_columns)

# Optionally slice to middle 20 frames (10 each side)
x_train_3d = get_middle_frames(x_train_3d, n_frames_each_side=10)
x_test_3d  = get_middle_frames(x_test_3d,  n_frames_each_side=10)

print(x_train_3d.shape)  # (n_samples, 20, n_features)
print(x_test_3d.shape)



Reshaping: 92 samples, 173 frames, 39 features per frame
Reshaping: 11 samples, 173 frames, 39 features per frame
Output shape: (92, 20, 39)
Output shape: (11, 20, 39)
(92, 20, 39)
(11, 20, 39)


## Define functions

In [99]:
# Define dense squat classifying model
class SquatClassifierDense(nn.Module):
    def __init__(self, input_dim, hidden_layers: list, activation="relu", dropout=0.0):
        super().__init__()

        layers = []
        activations = {"relu": nn.ReLU(),
                       "tanh": nn.Tanh(),
                       "gelu": nn.GELU(),
                       "leaky_relu": nn.LeakyReLU()
                       }
        
        prev_size = input_dim

        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(activations[activation])

            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            
            prev_size = hidden_size

        # Output layer
        layers.append(nn.Linear(prev_size, 1))

        self.network = nn.Sequential(*layers)
        self.network.apply(init_weights)

    def forward(self, x):
        return self.network(x)


def build_dense_model(config, input_size):
    return SquatClassifierDense(
        input_dim=input_size,
        hidden_layers=config["layers"],
        activation=config["activation"],
        dropout=config["dropout"]
    ).to(device)


# Define initial weights and biases
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight) # good for ReLU
        nn.init.zeros_(m.bias)


# Compute metrics
def compute_metrics(y_true, probs, threshold=0.5):

    preds = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

    accuracy = accuracy_score(y_true, preds)
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)

    # AUC
    auc = roc_auc_score(y_true, probs)

    return {
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "auc": auc
    }

# Full batch since it's faster on my cpu
def train_one_model(model, config, x_train, y_train, x_val, y_val, loss_fn):

    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    epochs = config["epochs"]

    best_val_auc = 0
    best_state = None

    patience = 10
    epochs_no_improve = 0

    # Convergence tracking
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auc": []
    }

    for epoch in range(epochs):

        # Training
        model.train()
        optimizer.zero_grad()

        logits = model(x_train)
        train_loss = loss_fn(logits, y_train)

        train_loss.backward()
        optimizer.step()

        # Validation with auc as main metric
        model.eval()
        with torch.no_grad():
            val_logits = model(x_val)

            val_loss = loss_fn(val_logits, y_val)

            probs = torch.sigmoid(val_logits)

            val_auc = roc_auc_score(
            y_val.cpu().numpy().ravel(),
            probs.cpu().numpy().ravel()
            )

        # Store history for each epoch
        history["train_loss"].append(train_loss.item())
        history["val_loss"].append(val_loss.item())
        history["val_auc"].append(val_auc)

        # Early stopping
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    # Load best weights
    model.load_state_dict(best_state)

    return best_val_auc, model, history


# Cross validation
def cross_validate_model(config, X, y, loss_fn, input_size, n_splits=10):

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_auc_scores = []
    fold_histories = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        # Split data
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Scale for each fold
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val   = scaler.transform(X_val)

        # Convert to tensors
        X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
        X_val   = torch.tensor(X_val, dtype=torch.float32).to(device)
        y_train = torch.tensor(y_train, dtype=torch.float32).to(device).view(-1, 1)
        y_val = torch.tensor(y_val, dtype=torch.float32).to(device).view(-1, 1)

        # Build fresh model
        model = build_dense_model(config, input_size)

        # Train
        val_auc, model, history = train_one_model(
            model, config,
            X_train, y_train,
            X_val, y_val,
            loss_fn
        )

        fold_auc_scores.append(val_auc)
        fold_histories.append(history)


    # Aggregate results
    mean_auc = np.mean(fold_auc_scores)
    std_auc  = np.std(fold_auc_scores)

    return {
        "cv_mean_auc": mean_auc,
        "auc_std": std_auc,
        "fold_scores": fold_auc_scores
    }

## Functions for model handling

In [100]:
def load_champion_info(metadata_dir):
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None

# Changed to be auc based

def save_champion_model(champion_dir, metadata_dir, model, model_name, auc, recall, precision, accuracy, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "auc": float(auc),
        "recall": float(recall),
        "precision": float(precision),
        "accuracy": float(accuracy),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")


# Auc based
def update_champion(metadata_dir, champion_dir, model, model_name, auc, recall, precision, accuracy, hyperparameters):
    current = load_champion_info(metadata_dir)

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(champion_dir, metadata_dir, model, model_name, auc, recall, precision, accuracy, hyperparameters)

    elif auc > current["auc"]:
        print(f"New model is better (auc {auc} > {current['auc']})")
        save_champion_model(champion_dir, metadata_dir, model, model_name, auc, recall, precision, accuracy, hyperparameters)

    else:
        print(f"Model NOT better (auc {auc} < {current['auc']})")

# Cross validation grid search for tuning the model 

In [101]:
param_grid = {"layers": [[128, 64]],
              "lr": [0.005],
              "dropout": [0],
              "activation": ["relu"],
              "epochs": [100]
            }

trial = 0
best_auc = 0
best_config = None
loss_func = nn.BCEWithLogitsLoss()


# Make configuration of every existing combinations of grid values
for values in product(
    param_grid["layers"],
    param_grid["lr"],
    param_grid["dropout"],
    param_grid["activation"],
    param_grid["epochs"]
    ):

    config = {
        "layers": values[0],
        "lr": values[1],
        "dropout": values[2],
        "activation": values[3],
        "epochs": values[4]
        }

    print(f"\nTrial: {trial + 1}")
    print(config)

    # Extract the number of features
    input_size = x_train.shape[1]

    # Cross validation
    results = cross_validate_model(config, x_train, y_train, loss_func, input_size)

    auc_score = results["cv_mean_auc"]
    auc_std = results["auc_std"]
    fold_scores = results["fold_scores"]

    print(f"AUC fold scores: {fold_scores}")
    print(f"CV mean AUC: {auc_score} +- {auc_std}")

    # Update best config
    if auc_score > best_auc:
        best_auc = auc_score
        best_config = config
    
    trial += 1


Trial: 1
{'layers': [128, 64], 'lr': 0.005, 'dropout': 0, 'activation': 'relu', 'epochs': 100}
AUC fold scores: [1.0, 0.8333333333333334, 0.8333333333333334, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
CV mean AUC: 0.9666666666666668 +- 0.06666666666666665


# Evaluate on test and update champion (including test weights)